### reproducing double agents logic using checkpoints

In [1]:
import gc
import sys
from pathlib import Path

import dllm
import torch
from huggingface_hub import snapshot_download

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "README.md").exists() and (path / "code").exists()
)

sys.path[:0] = [
    str(ROOT / "code" / "src"),
    str(ROOT / "code" / "scripts"),
]

DEVICE = torch.device("cuda")

model_paths = {
    "tiny": snapshot_download(
        repo_id="dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1",
        revision="c8d24a3f4adaeef46881b450e1bf7d1005203bd7",
        local_files_only=True,
    ),
    "dream": snapshot_download(
        repo_id="Dream-org/Dream-v0-Instruct-7B",
        revision="05334cb9faaf763692dcf9d8737c642be2b2a6ae",
        local_files_only=True,
    ),
}

---

### tiny a2d

In [2]:
import phase5_hm_synth_tiny as tiny_exp

from hidden_messages.datasets.hm_synth import generate_chain_lookup_pair

tiny_model = dllm.utils.get_model(
    model_name_or_path=model_paths["tiny"],
    dtype="bfloat16",
    attn_implementation="eager",
).eval()

tiny_model.requires_grad_(False)

tiny_tokenizer = dllm.utils.get_tokenizer(
    model_name_or_path=model_paths["tiny"],
)

tiny_head = tiny_model.get_output_embeddings()

tiny_channel = tiny_exp.ReceiverChannel(
    model_dim=tiny_model.config.hidden_size,
    message_slots=4,
    message_dim=128,
    width=256,
    num_heads=4,
).to(DEVICE).eval()

tiny_channel.load_state_dict(
    torch.load(
        ROOT / "checkpoints" / "tiny_a2d_channel.pt",
        map_location=DEVICE,
        weights_only=True,
    ),
    strict=True,
)

<All keys matched successfully>

In [5]:
tiny_example, _ = generate_chain_lookup_pair(
    seed=1234,
    num_agents=2,
    hops=2,
    distractors_per_agent=2,
    answer_values=tuple("0123456789"),
)

print("Question:", tiny_example.question)
print("Agent 0 context:", tiny_example.private_contexts[0])
print("Agent 1 context:", tiny_example.private_contexts[1])
print("Target:", tiny_example.answer)

Question: What is the code associated with gorfyn6? Return only the code.
Agent 0 context: boral21 is linked to elbor4. gorfyn6 is linked to gorfyn16. jungor9 is linked to halcy17.
Agent 1 context: aldra10 is linked to boral11. iojun8 has code 0. halcy7 is linked to drael13. gorfyn16 is linked to iojun8.
Target: 0


In [6]:
tiny_tokens = tiny_exp.tokenize_split(
    [tiny_example],
    tiny_tokenizer,
    max_sequence_length=256,
)

tiny_hidden = tiny_exp.precompute_hidden(
    tiny_tokens,
    tiny_model,
    tiny_head,
    batch_size=2,
    device=DEVICE,
)

/home/roman/Desktop/hidden-messages-diffusion-agents-math/.venv/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


In [20]:
tiny_message = tiny_exp.encode_all_messages(
    tiny_channel,
    tiny_hidden,
    agent_id=1,
    normalized_time=1.0,
    batch_size=1,
    device=DEVICE,
)

assert tiny_message.shape == (1, 4, 128)
print("[agent 1] Message shape:", tiny_message.shape)

tiny_no_message = tiny_exp.evaluate_no_message(
    tiny_hidden,
    tiny_head,
    batch_size=1,
    device=DEVICE,
)

tiny_matched = tiny_exp.evaluate_incoming(
    tiny_channel,
    tiny_hidden,
    tiny_head,
    tiny_message[:, None, None],
    sender_id=1,
    normalized_time=1.0,
    batch_size=1,
    device=DEVICE,
)

tiny_zero = tiny_exp.evaluate_incoming(
    tiny_channel,
    tiny_hidden,
    tiny_head,
    torch.zeros_like(tiny_message)[:, None, None],
    sender_id=1,
    normalized_time=1.0,
    batch_size=1,
    device=DEVICE,
)

def decode_tiny(token_id):
    return tiny_tokenizer.decode([int(token_id)]).strip()

print("Target:     ", tiny_example.answer)
print("[agent 0] No message: ", decode_tiny(tiny_no_message[0]))
print("[agent 0] Matched M:  ", decode_tiny(tiny_matched[0]))
print("[agent 0] Zero M:     ", decode_tiny(tiny_zero[0]))

[agent 1] Message shape: torch.Size([1, 4, 128])
Target:      0
[agent 0] No message:  g
[agent 0] Matched M:   0
[agent 0] Zero M:      g


In [21]:
del (
    tiny_model,
    tiny_tokenizer,
    tiny_head,
    tiny_channel,
    tiny_tokens,
    tiny_hidden,
    tiny_message,
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

---

### dream 

In [22]:
from hidden_messages.communication import DenseLatentPrefixChannel

dream_model = dllm.utils.get_model(
    model_name_or_path=model_paths["dream"],
    dtype="bfloat16",
    attn_implementation="sdpa",
).eval()

dream_model.requires_grad_(False)

dream_tokenizer = dllm.utils.get_tokenizer(
    model_name_or_path=model_paths["dream"],
)

dream_decoder = dream_model.get_decoder()
dream_embedding = dream_model.get_input_embeddings()
dream_head = dream_model.get_output_embeddings()

dream_state = torch.load(
    ROOT / "checkpoints" / "qasc_dense_latent_prefix_channel.pt",
    map_location=DEVICE,
    weights_only=True,
)

dream_channel = DenseLatentPrefixChannel(
    model_dim=dream_model.config.hidden_size,
    num_classes=8,
    input_embedding_rms=float(dream_state["input_embedding_rms"]),
    message_slots=4,
    message_dim=128,
    width=512,
    classifier_width=512,
).to(DEVICE).eval()

dream_channel.load_state_dict(dream_state, strict=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<All keys matched successfully>

In [59]:
LABELS = tuple("ABCDEFGH")

question = """
Which material conducts electricity well?

Options:
(A) material_a
(B) material_b
(C) material_c
(D) material_d
(E) material_e
(F) material_f
(G) material_g
(H) material_h

Return only the option label.
""".strip()

private_contexts = [
    "Materials classified as metals usually conduct electricity well.",
    "Material_b is classified as a metal.",
]

assistant_prefix = "Answer:"
assistant_prefix_ids = dream_tokenizer.encode(
    assistant_prefix,
    add_special_tokens=False,
)

def make_dream_tokens(private_context):
    prompt = dream_tokenizer.apply_chat_template(
        [{
            "role": "user",
            "content": (
                f"{question}\n\n"
                f"Private evidence:\n{private_context}"
            ),
        }],
        add_generation_prompt=True,
        tokenize=True,
    )

    return tuple(
        list(map(int, prompt))
        + list(map(int, assistant_prefix_ids))
        + [int(dream_tokenizer.mask_token_id)]
    )

dream_token_rows = [
    make_dream_tokens(context)
    for context in private_contexts
]



In [60]:
@torch.inference_mode()
def dream_forward(token_rows, prefixes=None):
    pad_id = dream_tokenizer.pad_token_id or dream_tokenizer.eos_token_id
    width = max(map(len, token_rows))

    input_ids = torch.full(
        (len(token_rows), width),
        pad_id,
        dtype=torch.long,
        device=DEVICE,
    )
    attention = torch.zeros_like(input_ids, dtype=torch.bool)
    lengths = torch.tensor(
        list(map(len, token_rows)),
        device=DEVICE,
    )

    for row_id, tokens in enumerate(token_rows):
        input_ids[row_id, :len(tokens)] = torch.tensor(
            tokens,
            device=DEVICE,
        )
        attention[row_id, :len(tokens)] = True

    embeddings = dream_embedding(input_ids)

    # -1 соответствует checkpoint-конфигу Dream.
    output_positions = lengths - 2

    if prefixes is not None:
        slots = prefixes.shape[1]
        expanded = embeddings.new_zeros(
            embeddings.shape[0],
            embeddings.shape[1] + slots,
            embeddings.shape[2],
        )
        expanded_attention = attention.new_zeros(
            attention.shape[0],
            attention.shape[1] + slots,
        )

        insertion_positions = (
            lengths - 1 - len(assistant_prefix_ids)
        )

        for row_id in range(embeddings.shape[0]):
            position = int(insertion_positions[row_id])

            expanded[row_id, :position] = embeddings[row_id, :position]
            expanded[row_id, position:position + slots] = prefixes[row_id]
            expanded[row_id, position + slots:] = embeddings[row_id, position:]

            expanded_attention[row_id, :position] = attention[row_id, :position]
            expanded_attention[row_id, position:position + slots] = True
            expanded_attention[row_id, position + slots:] = attention[row_id, position:]

            output_positions[row_id] += slots

        embeddings = expanded
        attention = expanded_attention

    sdpa_mask = attention[:, None, None, :]
    outputs = dream_decoder(
        inputs_embeds=embeddings,
        attention_mask=sdpa_mask,
        use_cache=False,
        return_dict=True,
    )

    rows = torch.arange(len(token_rows), device=DEVICE)
    selected_hidden = outputs.last_hidden_state[
        rows,
        output_positions,
    ]

    label_ids = torch.tensor(
        [
            dream_tokenizer.encode(label, add_special_tokens=False)[0]
            for label in LABELS
        ],
        device=DEVICE,
    )

    logits = dream_head(selected_hidden)
    predictions = logits.index_select(-1, label_ids).argmax(-1)

    return selected_hidden, predictions

In [61]:
@torch.inference_mode()
def run_dream_agents(rounds=4, zero_messages=False):
    prefixes = None
    last_messages = None

    for round_id in range(rounds):
        states, local_predictions = dream_forward(
            dream_token_rows,
            prefixes,
        )

        with torch.autocast("cuda", dtype=torch.bfloat16):
            messages = dream_channel.encode(states)

            assert messages.shape == (2, 4, 128)

            # Agent 0 получает M от Agent 1 и наоборот.
            incoming = torch.stack(
                [messages[1], messages[0]],
                dim=0,
            )

            if zero_messages:
                incoming = torch.zeros_like(incoming)

            prefixes = dream_channel.project(incoming)

        last_messages = messages

        said = [LABELS[int(value)] for value in local_predictions]
        print(f"Round {round_id + 1}:")
        print("Agent A answer:", said[0], "| Agent B answer:", said[1], "\n")

    _, receiver_prediction = dream_forward(
        [dream_token_rows[0]],
        prefixes[:1],
    )

    return LABELS[int(receiver_prediction[0])], last_messages

In [62]:
_, dream_no_message = dream_forward([dream_token_rows[0]])

print("With communication:")
dream_matched, dream_messages = run_dream_agents(
    rounds=1,
    zero_messages=False,
)

print("Communication replaced with zeros")
dream_zero, _ = run_dream_agents(
    rounds=1,
    zero_messages=True,
)

print("Expected:   B")
print("No message:", LABELS[int(dream_no_message[0])])
print("Matched M: ", dream_matched)
print("Zero M:    ", dream_zero)
print("Message shape:", dream_messages.shape)

With communication:
Round 1:
Agent A answer: G | Agent B answer: B 

Communication replaced with zeros
Round 1:
Agent A answer: G | Agent B answer: B 

Expected:   B
No message: G
Matched M:  G
Zero M:     G
Message shape: torch.Size([2, 4, 128])
